# This is my FYP model detection, improved version of P1 Defence. 
# By Mahnoor Bibi 

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json

ROOT = Path.cwd()

candidate_a = ROOT / "crime.v8i.yolov8"
candidate_b = ROOT / "models" / "weaponDetectionModel" / "crime.v8i.yolov8"

if candidate_a.exists():
    DATASET_ROOT = candidate_a
elif candidate_b.exists():
    DATASET_ROOT = candidate_b
else:
    DATASET_ROOT = candidate_a

SPLITS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG", ".PNG"}
CLASS_NAMES = {0: "criminal", 1: "person", 2: "weapon"}

print("Notebook CWD:", ROOT)
print("Dataset root:", DATASET_ROOT)
print("Exists:", DATASET_ROOT.exists())

# Count trainingi and validation data elements 

In [ ]:
split_inventory = {}

for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    images = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix in IMAGE_EXTS]) if img_dir.exists() else []
    labels = sorted(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

    split_inventory[split] = {
        "images": images,
        "labels": labels,
        "image_count": len(images),
        "label_count": len(labels),
    }

inventory_view = {
    s: {
        "image_count": split_inventory[s]["image_count"],
        "label_count": split_inventory[s]["label_count"],
    }
    for s in SPLITS
}

print(json.dumps(inventory_view, indent=2))
print("Total images:", sum(v["image_count"] for v in inventory_view.values()))

In [ ]:
box_counts = Counter()
image_presence = Counter()
malformed_rows = []
invalid_class_rows = []
invalid_range_rows = []
empty_label_files = []

for split in SPLITS:
    for label_path in split_inventory[split]["labels"]:
        lines = label_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()

        if not lines:
            empty_label_files.append((split, str(label_path)))
            continue

        classes_in_image = set()

        for line_no, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                malformed_rows.append((split, str(label_path), line_no, line))
                continue

            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except Exception:
                malformed_rows.append((split, str(label_path), line_no, line))
                continue

            if cls not in CLASS_NAMES:
                invalid_class_rows.append((split, str(label_path), line_no, cls))
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                invalid_range_rows.append((split, str(label_path), line_no, (x, y, w, h)))
                continue

            box_counts[cls] += 1
            classes_in_image.add(cls)

        for cls in classes_in_image:
            image_presence[cls] += 1

summary_counts = {
    "box_counts": {CLASS_NAMES[k]: box_counts[k] for k in sorted(CLASS_NAMES)},
    "image_presence": {CLASS_NAMES[k]: image_presence[k] for k in sorted(CLASS_NAMES)},
    "empty_label_files": len(empty_label_files),
    "malformed_rows": len(malformed_rows),
    "invalid_class_rows": len(invalid_class_rows),
    "invalid_range_rows": len(invalid_range_rows),
}

print(json.dumps(summary_counts, indent=2))

# Balancing the dataset 
# Validating dataset if any invalid rows or column exist 
# bounding boxes count

In [ ]:
def imbalance_status(max_count, min_count):
    if min_count == 0:
        return "severe"
    ratio = max_count / min_count
    if ratio > 2.5:
        return "severe"
    if ratio > 1.5:
        return "mild"
    return "ok"

box_vals = [box_counts[k] for k in sorted(CLASS_NAMES)]
img_vals = [image_presence[k] for k in sorted(CLASS_NAMES)]

box_status = imbalance_status(max(box_vals), min(box_vals)) if box_vals else "n/a"
img_status = imbalance_status(max(img_vals), min(img_vals)) if img_vals else "n/a"

ratios = {
    "box_level_max_min_ratio": (max(box_vals) / min(box_vals)) if min(box_vals) > 0 else None,
    "image_level_max_min_ratio": (max(img_vals) / min(img_vals)) if min(img_vals) > 0 else None,
    "box_imbalance_status": box_status,
    "image_presence_imbalance_status": img_status,
}

print(json.dumps(ratios, indent=2))

In [ ]:
split_stats = {}
for split in SPLITS:
    n = split_inventory[split]["image_count"]
    split_stats[split] = {
        "count": n,
    }

total = sum(split_stats[s]["count"] for s in SPLITS)
for split in SPLITS:
    split_stats[split]["pct"] = round((split_stats[split]["count"] / total) * 100, 2) if total else 0.0

target = {
    "train_70": int(total * 0.70),
    "valid_20": int(total * 0.20),
    "test_10": total - int(total * 0.70) - int(total * 0.20),
}

print("Current split:")
print(json.dumps(split_stats, indent=2))
print("Target counts (70/20/10):")
print(json.dumps(target, indent=2))

# checking split percentage 

# our target is 70 20 10 
# train valid test 

# finally we have balance state mild : we will work with that

In [ ]:
def sha1_file(path: Path, chunk_size: int = 1 << 20):
    h = hashlib.sha1()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

hash_index = defaultdict(list)
stem_index = defaultdict(list)

for split in SPLITS:
    for img in split_inventory[split]["images"]:
        digest = sha1_file(img)
        hash_index[digest].append((split, str(img)))
        stem_index[img.stem.lower()].append((split, str(img)))

exact_leaks = [v for v in hash_index.values() if len({x[0] for x in v}) > 1]
stem_leaks = [v for v in stem_index.values() if len({x[0] for x in v}) > 1]

print("Exact cross-split duplicates:", len(exact_leaks))
print("Potential cross-split same-stem leaks:", len(stem_leaks))

if exact_leaks:
    print("Sample exact leak:", exact_leaks[0][:3])
if stem_leaks:
    print("Sample stem leak:", stem_leaks[0][:3])

In [ ]:
audit_report = {
    "dataset_root": str(DATASET_ROOT),
    "total_images": total,
    "split_counts": {s: split_stats[s]["count"] for s in SPLITS},
    "split_percentages": {s: split_stats[s]["pct"] for s in SPLITS},
    "target_70_20_10": target,
    "box_counts": {CLASS_NAMES[k]: box_counts[k] for k in sorted(CLASS_NAMES)},
    "image_presence_counts": {CLASS_NAMES[k]: image_presence[k] for k in sorted(CLASS_NAMES)},
    "quality_flags": {
        "empty_label_files": len(empty_label_files),
        "malformed_rows": len(malformed_rows),
        "invalid_class_rows": len(invalid_class_rows),
        "invalid_range_rows": len(invalid_range_rows),
    },
    "leakage_flags": {
        "exact_cross_split_duplicates": len(exact_leaks),
        "same_stem_cross_split_potential": len(stem_leaks),
    },
}

out_path = DATASET_ROOT / "phase1_audit_report.json"
out_path.write_text(json.dumps(audit_report, indent=2), encoding="utf-8")

print("Saved:", out_path)
print(json.dumps(audit_report, indent=2))

In [ ]:
import random
import math

report_path = DATASET_ROOT / "phase1_audit_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

box_counts_report = report.get("box_counts", {})
max_box = max(box_counts_report.values()) if box_counts_report else 0
min_box = min(box_counts_report.values()) if box_counts_report else 0
box_ratio = (max_box / min_box) if min_box else float("inf")

imbalance_state = "severe" if box_ratio > 2.5 else ("mild" if box_ratio > 1.5 else "ok")
proceed_annotation_audit = imbalance_state in {"ok", "mild"}

print("Imbalance state:", imbalance_state)
print("Proceed with annotation-quality phase:", proceed_annotation_audit)

# Random spot check sampling

In [ ]:
def parse_label_file(label_path: Path):
    rows = []
    lines = label_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    for i, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            continue
        try:
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except Exception:
            continue
        rows.append((i, cls, x, y, w, h))
    return rows


def to_xyxy(x, y, w, h):
    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return (inter / union) if union > 0 else 0.0

In [ ]:
random.seed(42)

sample_total = 300
split_counts = report["split_counts"]

alloc = {}
remaining = sample_total
for idx, split in enumerate(SPLITS):
    if idx < len(SPLITS) - 1:
        n = round(sample_total * (split_counts[split] / report["total_images"]))
        alloc[split] = n
        remaining -= n
    else:
        alloc[split] = remaining

spotcheck_rows = []
for split in SPLITS:
    images = split_inventory[split]["images"]
    labels = {p.stem: p for p in split_inventory[split]["labels"]}
    k = min(alloc[split], len(images))
    chosen = random.sample(images, k) if k else []
    for img in chosen:
        lbl = labels.get(img.stem)
        anns = parse_label_file(lbl) if lbl and lbl.exists() else []
        cls_set = sorted({a[1] for a in anns})
        spotcheck_rows.append({
            "split": split,
            "image": str(img),
            "label": str(lbl) if lbl else "",
            "num_boxes": len(anns),
            "classes": [CLASS_NAMES.get(c, str(c)) for c in cls_set],
            "has_weapon": 2 in cls_set,
        })

spotcheck_path = DATASET_ROOT / "phase1_spotcheck_300.json"
spotcheck_path.write_text(json.dumps(spotcheck_rows, indent=2), encoding="utf-8")

print("Spot-check samples:", len(spotcheck_rows))
print("Saved:", spotcheck_path)
print("Per-split allocation:", alloc)

# checking for dublicates in dataset 
# checing for larger  bounding boexs 

In [ ]:
duplicate_overlap_flags = []
size_flags = []
keyword_missing_weapon_flags = []

weapon_keywords = ("gun", "knife", "pistol", "weapon", "rifle", "revolver")

for split in SPLITS:
    labels = split_inventory[split]["labels"]
    image_map = {p.stem: p for p in split_inventory[split]["images"]}

    for lbl in labels:
        anns = parse_label_file(lbl)
        if not anns:
            continue

        classes = [a[1] for a in anns]
        stem_lower = lbl.stem.lower()
        if any(k in stem_lower for k in weapon_keywords) and 2 not in classes:
            keyword_missing_weapon_flags.append({
                "split": split,
                "label": str(lbl),
                "image": str(image_map.get(lbl.stem, "")),
                "classes_present": sorted(set(classes)),
            })

        for idx, (_, cls, x, y, w, h) in enumerate(anns):
            area = w * h
            if area < 0.0005 or area > 0.80:
                size_flags.append({
                    "split": split,
                    "label": str(lbl),
                    "line_index": idx + 1,
                    "class_id": cls,
                    "bbox": [x, y, w, h],
                    "area": area,
                })

        for i in range(len(anns)):
            _, c1, x1, y1, w1, h1 = anns[i]
            b1 = to_xyxy(x1, y1, w1, h1)
            for j in range(i + 1, len(anns)):
                _, c2, x2, y2, w2, h2 = anns[j]
                if c1 != c2:
                    continue
                b2 = to_xyxy(x2, y2, w2, h2)
                ov = iou_xyxy(b1, b2)
                if ov > 0.9:
                    duplicate_overlap_flags.append({
                        "split": split,
                        "label": str(lbl),
                        "class_id": c1,
                        "pair": [i + 1, j + 1],
                        "iou": ov,
                    })

print("Duplicate overlap flags:", len(duplicate_overlap_flags))
print("Extreme bbox-size flags:", len(size_flags))
print("Keyword-based missing-weapon flags:", len(keyword_missing_weapon_flags))

In [ ]:
annotation_audit_report = {
    "proceed_annotation_audit": proceed_annotation_audit,
    "imbalance_state": imbalance_state,
    "spotcheck_sample_count": len(spotcheck_rows),
    "spotcheck_manifest": str(spotcheck_path),
    "flags_summary": {
        "duplicate_overlap_flags": len(duplicate_overlap_flags),
        "extreme_bbox_size_flags": len(size_flags),
        "keyword_missing_weapon_flags": len(keyword_missing_weapon_flags),
    },
    "flags_preview": {
        "duplicate_overlap_first10": duplicate_overlap_flags[:10],
        "extreme_bbox_size_first10": size_flags[:10],
        "keyword_missing_weapon_first10": keyword_missing_weapon_flags[:10],
    },
}

annotation_report_path = DATASET_ROOT / "phase1_annotation_quality_report.json"
annotation_report_path.write_text(json.dumps(annotation_audit_report, indent=2), encoding="utf-8")

print("Saved:", annotation_report_path)
print(json.dumps(annotation_audit_report["flags_summary"], indent=2))

# cheeckiing for images quality scan 
# total images ~ 7000

In [ ]:
import cv2
import numpy as np

all_split_images = []
for split in SPLITS:
    all_split_images.extend((split, p) for p in split_inventory[split]["images"])

print("Total images for quality scan:", len(all_split_images))

In [ ]:
def image_quality_metrics(img_bgr: np.ndarray):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(gray.mean())
    contrast = float(gray.std())
    h, w = gray.shape
    aspect_ratio = float(max(h, w) / max(1, min(h, w)))
    return {
        "h": int(h),
        "w": int(w),
        "lap_var": lap_var,
        "brightness": brightness,
        "contrast": contrast,
        "aspect_ratio": aspect_ratio,
    }

quality_rows = []
unreadable = []

for split, img_path in all_split_images:
    img = cv2.imread(str(img_path))
    if img is None:
        unreadable.append({"split": split, "image": str(img_path), "reason": "unreadable"})
        continue
    m = image_quality_metrics(img)
    quality_rows.append({
        "split": split,
        "image": str(img_path),
        **m,
    })

quality_path = DATASET_ROOT / "phase2_quality_metrics.json"
quality_path.write_text(json.dumps(quality_rows, indent=2), encoding="utf-8")

print("Saved:", quality_path)
print("Readable images:", len(quality_rows))
print("Unreadable images:", len(unreadable))

In [ ]:
HIGH_THRESHOLDS = {
    "very_dark": 20.0,
    "very_low_contrast": 10.0,
    "very_blurry": 18.0,
    "extreme_aspect_ratio": 3.5,
}

MEDIUM_THRESHOLDS = {
    "dark": 35.0,
    "low_contrast": 18.0,
    "blurry": 40.0,
}

high_noise = []
medium_noise = []

for row in quality_rows:
    reasons_high = []
    reasons_medium = []

    if row["brightness"] < HIGH_THRESHOLDS["very_dark"] and row["contrast"] < HIGH_THRESHOLDS["very_low_contrast"]:
        reasons_high.append("very_dark_and_low_contrast")
    if row["lap_var"] < HIGH_THRESHOLDS["very_blurry"] and row["contrast"] < HIGH_THRESHOLDS["very_low_contrast"]:
        reasons_high.append("very_blurry_and_low_contrast")
    if row["aspect_ratio"] > HIGH_THRESHOLDS["extreme_aspect_ratio"]:
        reasons_high.append("extreme_aspect_ratio")

    if row["brightness"] < MEDIUM_THRESHOLDS["dark"]:
        reasons_medium.append("dark")
    if row["contrast"] < MEDIUM_THRESHOLDS["low_contrast"]:
        reasons_medium.append("low_contrast")
    if row["lap_var"] < MEDIUM_THRESHOLDS["blurry"]:
        reasons_medium.append("blurry")

    if reasons_high:
        high_noise.append({**row, "reasons": sorted(set(reasons_high))})
    elif len(set(reasons_medium)) >= 2:
        medium_noise.append({**row, "reasons": sorted(set(reasons_medium))})

if unreadable:
    for bad in unreadable:
        high_noise.append({
            "split": bad["split"],
            "image": bad["image"],
            "h": None,
            "w": None,
            "lap_var": None,
            "brightness": None,
            "contrast": None,
            "aspect_ratio": None,
            "reasons": ["unreadable"],
        })

noise_report = {
    "high_noise_count": len(high_noise),
    "medium_noise_count": len(medium_noise),
    "high_noise_preview": high_noise[:30],
    "medium_noise_preview": medium_noise[:30],
}

noise_report_path = DATASET_ROOT / "phase2_noise_report.json"
noise_report_path.write_text(json.dumps(noise_report, indent=2), encoding="utf-8")

high_list_path = DATASET_ROOT / "phase2_high_noise_candidates.json"
high_list_path.write_text(json.dumps(high_noise, indent=2), encoding="utf-8")

print("Saved:", noise_report_path)
print("Saved:", high_list_path)
print(json.dumps({"high_noise_count": len(high_noise), "medium_noise_count": len(medium_noise)}, indent=2))

In [ ]:
from shutil import move

APPLY_HIGH_NOISE_QUARANTINE = False

quarantine_root = DATASET_ROOT / "_quarantine_high_noise"

moved = []
missing_pairs = []

for row in high_noise:
    img = Path(row["image"])
    split = row["split"]
    lbl = DATASET_ROOT / split / "labels" / f"{img.stem}.txt"

    q_img_dir = quarantine_root / split / "images"
    q_lbl_dir = quarantine_root / split / "labels"

    if APPLY_HIGH_NOISE_QUARANTINE:
        q_img_dir.mkdir(parents=True, exist_ok=True)
        q_lbl_dir.mkdir(parents=True, exist_ok=True)

        if img.exists():
            move(str(img), str(q_img_dir / img.name))
        if lbl.exists():
            move(str(lbl), str(q_lbl_dir / lbl.name))
        else:
            missing_pairs.append(str(lbl))

    moved.append({
        "split": split,
        "image": str(img),
        "label": str(lbl),
        "reasons": row["reasons"],
    })

quarantine_manifest = {
    "apply_high_noise_quarantine": APPLY_HIGH_NOISE_QUARANTINE,
    "high_noise_candidates": len(high_noise),
    "medium_noise_kept": len(medium_noise),
    "moved_or_planned": len(moved),
    "missing_labels_when_apply_true": len(missing_pairs),
    "items_preview": moved[:40],
}

quarantine_manifest_path = DATASET_ROOT / "phase2_high_noise_quarantine_manifest.json"
quarantine_manifest_path.write_text(json.dumps(quarantine_manifest, indent=2), encoding="utf-8")

print("Saved:", quarantine_manifest_path)
print(json.dumps({
    "apply_high_noise_quarantine": APPLY_HIGH_NOISE_QUARANTINE,
    "high_noise_candidates": len(high_noise),
    "medium_noise_kept": len(medium_noise),
}, indent=2))